# 脳波デコーディング体験 ② — EEGNet（深層学習）

ノート ① では「前処理 → CSP（特徴抽出）→ 分類器」と段階を分けました。
ここでは **EEGNet** という小型の畳み込みニューラルネットで、
前処理済み波形から **end-to-end**（特徴設計なし）で左右を当てます。

## EEGNet とは
EEG 用に設計されたコンパクトな CNN（Lawhern et al., 2018）。主役は 3 種類の畳み込み：
1. **時間方向の畳み込み** … 各チャンネルで帯域フィルタのようなものを学習
2. **Depthwise 畳み込み（空間）** … チャンネル方向に重みを学習＝ CSP のような空間フィルタの役割
3. **Separable 畳み込み** … 時間パターンを効率よくまとめる

CSP+LDA を「人間が設計した空間フィルタ＋線形分類」とすれば、EEGNet は
それを **データから丸ごと学習** する版、と捉えると分かりやすいです。

> ⚠️ 今回のデータは 100 トライアルと **とても小規模**。深層学習は本来
> 大量データで強みを発揮するので、古典手法（CSP）に精度で負けることも
> あります。「DL なら必ず勝つ」わけではない、というのも大事な学びです。

In [1]:
import mi_utils as U          # 先頭で KMP_DUPLICATE_LIB_OK を設定済み
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, cohen_kappa_score)

U.setup_japanese_font()  # グラフの日本語が豆腐(□)にならないように

# 再現性
SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)

# Apple Silicon なら MPS、無ければ CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device, "| torch", torch.__version__)

device: mps | torch 2.12.1


## 1. データの準備（窓に展開してデータ拡張）

実際のリアルタイム BCI は **2 秒の窓** を 0.5 秒ずつずらして分類して
います。同じ要領で 4 秒トライアルを 2 秒窓に切り分けると、サンプル数が
増えて小規模データでの学習が安定します。

In [2]:
# μ帯(8-13Hz)に絞る：実験でリーマン法と同様 μ帯が EEGNet でも有効
ds = U.load_dataset(bandpass=(8, 13))
Xw, yw, sess_w = U.make_windows(ds, window_s=2.0, stride_s=0.5)
print("窓データ:", Xw.shape, "(窓数, チャンネル, サンプル)")
print("ラベル内訳:", np.bincount(yw), "| セッション:", np.bincount(sess_w))

n_ch, n_time = Xw.shape[1], Xw.shape[2]

窓データ: (500, 14, 256) (窓数, チャンネル, サンプル)
ラベル内訳: [250 250] | セッション: [100 100 100 100 100]


## 2. EEGNet の実装

論文の標準構成（F1=8, D=2, F2=16）を、14ch・256 サンプル（2 秒）・2 クラス
向けに用意します。各層が何をしているかコメントを付けています。

In [3]:
class EEGNet(nn.Module):
    def __init__(self, n_ch, n_time, n_classes=2, F1=8, D=2, F2=16,
                 kern_len=64, p_drop=0.5):
        super().__init__()
        # --- ブロック1: 時間畳み込み + 空間(depthwise)畳み込み ---
        # 時間方向の畳み込み = 学習可能な帯域フィルタ群（F1 枚）
        self.temporal = nn.Sequential(
            nn.Conv2d(1, F1, (1, kern_len), padding=(0, kern_len // 2), bias=False),
            nn.BatchNorm2d(F1),
        )
        # depthwise: チャンネル(空間)方向にまとめる ≒ CSP 的な空間フィルタ
        self.spatial = nn.Sequential(
            nn.Conv2d(F1, F1 * D, (n_ch, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),      # 時間方向にダウンサンプル
            nn.Dropout(p_drop),
        )
        # --- ブロック2: separable 畳み込み（時間パターンの集約） ---
        self.separable = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(p_drop),
        )
        # 出力サイズを動的に求めて全結合層を作る
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_ch, n_time)
            feat_dim = self.separable(self.spatial(self.temporal(dummy))).numel()
        self.classify = nn.Linear(feat_dim, n_classes)

    def forward(self, x):                 # x: (N, 1, n_ch, n_time)
        x = self.temporal(x)
        x = self.spatial(x)
        x = self.separable(x)
        x = x.flatten(1)
        return self.classify(x)

# 形状チェック
_m = EEGNet(n_ch, n_time)
print(_m(torch.zeros(2, 1, n_ch, n_time)).shape, "(バッチ, クラス)")
print("パラメータ数:", sum(p.numel() for p in _m.parameters()))

torch.Size([2, 2]) (バッチ, クラス)
パラメータ数: 1586


## 3. 学習と評価（Leave-One-Session-Out）

ノート ① と同じく、汎化性能を見るため **セッションをまたいで** 評価します。
各 fold で:
1. テスト用に 1 セッションを除外
2. 残りで **チャンネルごとの z 標準化** の統計量を計算（訓練データのみ）
3. EEGNet を学習し、除外セッションで精度を測る

In [4]:
def standardize(train, *others):
    # チャンネルごとに平均・分散を訓練データから求めて適用
    mean = train.mean(axis=(0, 2), keepdims=True)
    std = train.std(axis=(0, 2), keepdims=True) + 1e-7
    return [( (a - mean) / std ) for a in (train, *others)]

def to_tensor(X, y):
    Xt = torch.tensor(X[:, None, :, :], dtype=torch.float32)  # (N,1,C,T)
    yt = torch.tensor(y, dtype=torch.long)
    return Xt, yt

def train_one_fold(Xtr, ytr, Xte, yte, epochs=120, batch=32, lr=1e-3,
                   weight_decay=1e-4, use_scheduler=True, verbose=False):
    Xtr, Xte = standardize(Xtr, Xte)
    Xtr_t, ytr_t = to_tensor(Xtr, ytr)
    Xte_t, yte_t = to_tensor(Xte, yte)

    model = EEGNet(n_ch, n_time).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    # CosineAnnealing: LR を余弦関数的に減衰させ、収束を安定させる
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs) if use_scheduler else None
    lossf = nn.CrossEntropyLoss()
    ds_tr = torch.utils.data.TensorDataset(Xtr_t, ytr_t)
    loader = torch.utils.data.DataLoader(ds_tr, batch_size=batch, shuffle=True)

    history = []
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = lossf(model(xb), yb)
            loss.backward(); opt.step()
        if scheduler: scheduler.step()
        # エポックごとのテスト精度を記録（学習曲線の可視化用）
        model.eval()
        with torch.no_grad():
            pred = model(Xte_t.to(device)).argmax(1).cpu().numpy()
        acc = (pred == yte).mean()
        history.append(acc)
        if verbose and (ep + 1) % 20 == 0:
            print(f"  epoch {ep+1:3d} | test acc {acc:.2f}")
    # 確率スコアも返す（AUC 計算用）
    model.eval()
    with torch.no_grad():
        proba = torch.softmax(model(Xte_t.to(device)), dim=1).cpu().numpy()
    return pred, proba, np.array(history)

In [5]:
logo = LeaveOneGroupOut()
fold_acc, all_true, all_pred, all_proba = [], [], [], []
first_history = None

for k, (tr, te) in enumerate(logo.split(Xw, yw, groups=sess_w)):
    pred, proba, hist = train_one_fold(Xw[tr], yw[tr], Xw[te], yw[te],
                                       epochs=120, use_scheduler=True, verbose=(k == 0))
    acc = (pred == yw[te]).mean()
    fold_acc.append(acc)
    all_true.append(yw[te]); all_pred.append(pred); all_proba.append(proba)
    if first_history is None:
        first_history = hist
    print(f"fold {k} (test=session {sess_w[te][0]}): acc {acc:.3f}")

yt = np.concatenate(all_true)
yp = np.concatenate(all_pred)
yprob = np.concatenate(all_proba)

auc   = roc_auc_score(yt, yprob[:, 1])
kappa = cohen_kappa_score(yt, yp)
print(f"\nEEGNet  LOSO  acc={np.mean(fold_acc):.3f}±{np.std(fold_acc):.3f}"
      f"  AUC={auc:.3f}  κ={kappa:.3f}")
print("（チャンスレベル: acc/AUC=0.50, κ=0.00）")

  epoch  20 | test acc 0.51


  epoch  40 | test acc 0.52


  epoch  60 | test acc 0.50


  epoch  80 | test acc 0.50


  epoch 100 | test acc 0.53


  epoch 120 | test acc 0.56


fold 0 (test=session 0): acc 0.560


fold 1 (test=session 1): acc 0.540


fold 2 (test=session 2): acc 0.550


fold 3 (test=session 3): acc 0.550


fold 4 (test=session 4): acc 0.760

EEGNet  LOSO  acc=0.592±0.084  AUC=0.624  κ=0.184
（チャンスレベル: acc/AUC=0.50, κ=0.00）


## 4. 結果の可視化

学習曲線（fold 0）と、全 fold をまとめた混同行列を見ます。

In [6]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(first_history)
ax1.set(xlabel="epoch", ylabel="test accuracy", title="学習曲線 (fold 0)", ylim=(0, 1))
ax1.axhline(0.5, color="gray", ls="--", label="chance")
ax1.legend()

cm = confusion_matrix(yt, yp)
ConfusionMatrixDisplay(cm, display_labels=[U.CLASS_NAMES[0], U.CLASS_NAMES[1]]).plot(
    cmap="Oranges", values_format="d", ax=ax2, colorbar=False)
ax2.set_title(f"EEGNet  acc={(yt==yp).mean():.2f}  AUC={auc:.2f}  κ={kappa:.2f}")
plt.tight_layout(); plt.show()

/var/folders/7_/5h8t97xx06q690jwmw8188kr0000gn/T/ipykernel_51226/1876504919.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. まとめと演習

**改善のポイント（ベースラインからの変更点）**
| 変更 | 効果 |
|---|---|
| `bandpass=(8,13)` μ帯に絞る | 運動想起の ERD が最も強い帯域に集中 |
| `epochs=120`（60→120）| 小バッチで収束が遅いため、十分な反復が必要 |
| `CosineAnnealingLR` | LR を余弦関数で滑らかに減衰させ、収束を安定化 |
| `weight_decay=1e-4` | L2 正則化でわずかに過学習を抑制 |

**まとめ**
- μ帯フィルタ + LR スケジューラで EEGNet の安定性が向上する。
- depthwise 畳み込みは CSP のような **空間フィルタ** を学習しており、
  古典手法と発想は地続き。
- それでも LOSO でリーマン手法（FgMDM ~0.64）に追いつくのは難しい。
  **DL が真に強みを発揮するにはデータ量が 1 桁以上必要**。

**演習**
1. `use_scheduler=False` に戻すと学習曲線はどう変わる？
2. `epochs` を 60 / 200 に変えると精度は？（過学習 vs 未学習の境界を探そう）
3. `make_windows(window_s=..., stride_s=...)` で窓の長さ・刻みを変えると？
4. ノート ① の FgMDM と今回の EEGNet の精度を並べて比べよう。
   「どちらが良い」より「なぜ差が出るのか」を考えてみよう。